# Lab 18 — OpenTelemetry portable tracing

Instrument the same Lab 14-style supervisor agent from Lab 17, this time with OpenTelemetry's GenAI semantic conventions. Three exporters: console (immediate dev visibility), OTLP→LangSmith (same UI as Lab 17), and optionally OTLP→Jaeger.

Lab 17 showed the LangSmith-native path; Lab 18 shows the vendor-neutral one. Same agent, different instrumentation layer.

> 📖 Required reading: [`concepts/evaluation/opentelemetry-genai-conventions.md`](../../concepts/evaluation/opentelemetry-genai-conventions.md), [`concepts/evaluation/platform-fanout-and-portability.md`](../../concepts/evaluation/platform-fanout-and-portability.md).
> ⬅️ Recommended (not strictly required): [Lab 17](../17-langsmith-trace-ingestion/) — the LangSmith-native counterpart for direct comparison.
> 🛠 Need a LangSmith account (free tier sufficient, same as Lab 17). Optional: docker for the Step 7 Jaeger demonstration.
> ⏱ Run time: 90-110 min including reading.


## Step 0: Setup

Install the OpenTelemetry SDK + OTLP exporter + OpenAI auto-instrumentation. Set env vars: `OTEL_SERVICE_NAME`, `OTEL_EXPORTER_OTLP_ENDPOINT` (pointing to LangSmith's `/otel`), `OTEL_EXPORTER_OTLP_HEADERS` (with the LangSmith API key), and `OTEL_SEMCONV_STABILITY_OPT_IN=gen_ai_latest_experimental` for v1.37+ aggregated attributes.

The setup cell installs three packages. Pinning them in `pyproject.toml` is a hygiene-batch task.

In [ ]:
# Install dependencies (if not already installed)
# !pip install --quiet opentelemetry-sdk opentelemetry-exporter-otlp opentelemetry-instrumentation-openai

import os
import json
import pathlib
from typing import Literal

from dotenv import load_dotenv

# Load .env from repo root
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

# OTel configuration — set the service identity and the OTLP target
os.environ["OTEL_SERVICE_NAME"] = "lab-18-otel-portable"
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://api.smith.langchain.com/otel"

LANGSMITH_API_KEY = os.environ.get("LANGSMITH_API_KEY", "")
assert LANGSMITH_API_KEY, "Set LANGSMITH_API_KEY in your .env or shell"

os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = (
    f"x-api-key={LANGSMITH_API_KEY},Langsmith-Project=lab-18-opentelemetry"
)

# Opt in to v1.37+ aggregated attributes (gen_ai.input.messages, etc.)
os.environ["OTEL_SEMCONV_STABILITY_OPT_IN"] = "gen_ai_latest_experimental"

# Verify
print(f"OTEL_SERVICE_NAME: {os.environ['OTEL_SERVICE_NAME']}")
print(f"OTEL_EXPORTER_OTLP_ENDPOINT: {os.environ['OTEL_EXPORTER_OTLP_ENDPOINT']}")
print(f"OTEL_SEMCONV_STABILITY_OPT_IN: {os.environ['OTEL_SEMCONV_STABILITY_OPT_IN']}")


## Step 1: Inline minimal Lab 14 agent

Same compact supervisor pattern as Lab 17 — one supervisor, one stub researcher, one writer. The agent itself isn't the point of this lab; the instrumentation is.

We use the OpenAI SDK directly (not via LangChain's wrapper) to make the OpenAIInstrumentor auto-instrumentation concrete. Anthropic users swap to `opentelemetry-instrumentation-anthropic`; the pattern is identical.

In [ ]:
from openai import OpenAI

# Provider-agnostic but using OpenAI directly for the lab
client = OpenAI()
MODEL = "gpt-4o-mini"


def researcher(question: str) -> dict:
    """Stub researcher — fixed findings to keep the lab deterministic."""
    return {
        "findings": "MCP is an open standard for connecting LLMs to tools [1]. "
                    "Recent developments include broader production adoption [2].",
        "citations": [
            {"url": "https://example.org/mcp-overview", "title": "MCP Overview"},
            {"url": "https://example.com/mcp-2026", "title": "MCP in 2026"},
        ],
    }


def writer(findings: str) -> str:
    """Writer — composes the final answer from findings."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system",
             "content": "Compose a 100-word summary from these findings. "
                        "Preserve citation markers [1], [2] exactly."},
            {"role": "user", "content": f"FINDINGS:\n{findings}"},
        ],
        temperature=0,
    )
    return response.choices[0].message.content


def supervisor_route(task: str, completed: dict) -> str:
    """Supervisor LLM picks the next action: 'researcher', 'writer', or 'done'."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system",
             "content": "You coordinate two workers. Respond with ONLY one word: "
                        "'researcher' to call the researcher, 'writer' to call the "
                        "writer, or 'done' to finalize. Call researcher first, "
                        "then writer, then done."},
            {"role": "user", "content": f"Task: {task}\nCompleted so far: {list(completed.keys())}"},
        ],
        temperature=0,
        max_tokens=10,
    )
    return response.choices[0].message.content.strip().lower()


def run_agent(task: str, max_steps: int = 4) -> dict:
    """Top-level agent loop. Routes via supervisor; calls workers as directed."""
    completed = {}
    for _step in range(max_steps):
        action = supervisor_route(task, completed)
        if action == "researcher":
            completed["researcher"] = researcher(task)
        elif action == "writer":
            completed["writer"] = writer(completed.get("researcher", {}).get("findings", ""))
        elif action == "done":
            break
    return completed


print(f"Agent loaded. Model: {MODEL}")


## Step 2: Configure the `TracerProvider` with a `Resource`

The `TracerProvider` is the single source of identity for spans this process emits. The `Resource` attaches process-level attributes (service name, deployment environment) to every span automatically — you don't repeat them per span.

This is the OTel equivalent of the "tag every trace at the root with `user_id`/`tenant_id`/`agent_version`" pattern. Set it once, propagated to every span.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.resources import Resource, SERVICE_NAME, DEPLOYMENT_ENVIRONMENT

resource = Resource.create({
    SERVICE_NAME: "lab-18-otel-portable",
    DEPLOYMENT_ENVIRONMENT: "development",
    "agent.framework": "custom",  # not LangChain in this lab
    "agent.version": "0.1.0",
})

provider = TracerProvider(resource=resource)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer(__name__)
print("TracerProvider configured.")
print(f"  Service: {resource.attributes.get(SERVICE_NAME)}")
print(f"  Environment: {resource.attributes.get(DEPLOYMENT_ENVIRONMENT)}")


## Step 3: Add the `ConsoleSpanExporter`

Console output during development is non-negotiable. Watching spans print as the agent runs catches bugs (missing attributes, wrong span kind, unexpected nesting) faster than checking a UI.

`SimpleSpanProcessor` exports each span immediately — good for the console because you see it in real time. Production exporters use `BatchSpanProcessor` (batches and flushes asynchronously) because the network latency would otherwise serialize span emission with the main thread.

In [ ]:
from opentelemetry.sdk.trace.export import (
    SimpleSpanProcessor, BatchSpanProcessor, ConsoleSpanExporter,
)

# Console exporter for dev visibility — immediate, not batched
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))

# Test it with a manual span
with tracer.start_as_current_span("setup_test_span") as span:
    span.set_attribute("test.lab", 18)
    span.set_attribute("test.step", 3)

print("\n→ The span above printed as a JSON object. That's the raw OTel format.")
print("→ Every backend ingests this same shape; only the UI rendering differs.")


## Step 4: Manual `gen_ai.chat` spans with `SpanKind.CLIENT`

The critical attribute: `SpanKind.CLIENT` on LLM calls. Without it, APMs like Datadog group LLM spans under "internal database operations" in service maps. Two minutes to fix; hours to debug. This is the cheap-to-get-right, expensive-to-debug attribute.

The canonical `gen_ai.*` attribute set: `gen_ai.system`, `gen_ai.request.model`, `gen_ai.usage.input_tokens`, `gen_ai.usage.output_tokens`. Setting these makes the span queryable in any GenAI-aware backend.

In [ ]:
from opentelemetry.trace import SpanKind, Status, StatusCode


def call_llm_with_manual_span(prompt: str, model: str = MODEL) -> str:
    """Wrap an OpenAI call with a manual gen_ai.chat span. Demonstrates the
    canonical attribute set and SpanKind=CLIENT discipline."""
    with tracer.start_as_current_span(
        "gen_ai.chat",
        kind=SpanKind.CLIENT,
    ) as span:
        # Set request attributes BEFORE the call
        span.set_attribute("gen_ai.system", "openai")
        span.set_attribute("gen_ai.request.model", model)
        span.set_attribute("gen_ai.operation.name", "chat")
        span.set_attribute("gen_ai.request.temperature", 0.0)

        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )

            # Set response attributes AFTER the call
            span.set_attribute("gen_ai.response.model", response.model)
            span.set_attribute("gen_ai.usage.input_tokens", response.usage.prompt_tokens)
            span.set_attribute("gen_ai.usage.output_tokens", response.usage.completion_tokens)
            span.set_status(Status(StatusCode.OK))

            return response.choices[0].message.content

        except Exception as e:
            span.set_status(Status(StatusCode.ERROR, str(e)))
            span.record_exception(e)
            raise


# Try it
result = call_llm_with_manual_span("What is OpenTelemetry in one sentence?")
print(f"\nResponse: {result[:200]}")


**What just printed**: a span emitted to the console showing the gen_ai attributes you set. In a real backend (LangSmith / Datadog / Jaeger) the same data renders with the appropriate UI affordances.

Why `SpanKind.CLIENT` matters: APM service-maps group spans by kind. `CLIENT` means "outgoing call to an external system" — the right grouping for LLM API calls. The default kind is `INTERNAL`, which means "code running inside this service" — wrong for LLM calls, wrong UX downstream.

## Step 5: Auto-instrument with `OpenAIInstrumentor`

The manual approach in Step 4 makes the attributes explicit. The auto-instrumented approach makes them automatic — `OpenAIInstrumentor().instrument()` monkey-patches the openai SDK so every chat.completions call emits the same gen_ai span without per-call decoration.

The pattern in production: auto-instrument the provider SDK (saves boilerplate); manual spans for the orchestration layer (agent boundaries, tool execution, custom retrievers) that auto-instrumentation doesn't reach.

In [ ]:
from opentelemetry.instrumentation.openai import OpenAIInstrumentor

# Idempotent — calling twice doesn't double-instrument
OpenAIInstrumentor().instrument()

# Now every openai.chat.completions.create call emits gen_ai spans automatically
result = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is OpenTelemetry in one sentence?"}],
    temperature=0,
)
print(f"\nResponse: {result.choices[0].message.content[:200]}")
print()
print("→ A gen_ai.chat span was auto-emitted. Same attributes as Step 4's manual span.")
print("→ For LangChain users, opentelemetry-instrumentation-langchain provides the equivalent")
print("  auto-instrumentation for LangChain + LangGraph nodes.")


## Step 6: Add the OTLP exporter pointing at LangSmith

This is where fanout starts. The console exporter from Step 3 stays attached. We add a second `BatchSpanProcessor` with an `OTLPSpanExporter` pointing at LangSmith's `/otel` endpoint. Spans now land in both places.

`BatchSpanProcessor` is the right choice for OTLP — it batches spans and flushes asynchronously, so the network latency doesn't serialize span emission with the main thread.

In [ ]:
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

# OTLP exporter targeting LangSmith
# Headers come from OTEL_EXPORTER_OTLP_HEADERS env var (set in Step 0)
otlp_exporter = OTLPSpanExporter()

# BatchSpanProcessor for production — batches and flushes async
provider.add_span_processor(BatchSpanProcessor(otlp_exporter))

print("OTLP exporter attached.")
print(f"  Endpoint: {os.environ['OTEL_EXPORTER_OTLP_ENDPOINT']}")
print("  Project: lab-18-opentelemetry (set via header)")
print()
print("→ Now every span lands in BOTH the console (immediate, this notebook)")
print("  AND LangSmith (after the BatchSpanProcessor flushes, ~5 seconds).")


In [ ]:
# Run the agent. Watch the console for spans; check LangSmith UI for the same.
task = "Research recent developments in MCP and write a 100-word summary."
result = run_agent(task)

print("\nAgent output:")
print("─" * 60)
print(result.get("writer", "[no writer output]")[:300])
print()
print("→ Open smith.langchain.com → project 'lab-18-opentelemetry'")
print("→ The same trace is in the LangSmith UI within ~5 seconds")


**Sample console output** (one span emission, abbreviated):

```json
{
  "name": "gen_ai.chat",
  "kind": "SpanKind.CLIENT",
  "attributes": {
    "gen_ai.system": "openai",
    "gen_ai.request.model": "gpt-4o-mini",
    "gen_ai.operation.name": "chat",
    "gen_ai.response.model": "gpt-4o-mini-2024-07-18",
    "gen_ai.usage.input_tokens": 156,
    "gen_ai.usage.output_tokens": 12
  },
  "resource": {
    "service.name": "lab-18-otel-portable",
    "deployment.environment": "development"
  }
}
```

**What you should see in LangSmith UI**: open the `lab-18-opentelemetry` project. The trace appears with the same attributes. The UI renders the call as a conversation; the console renders it as JSON. Same data, different views.

## Step 7 (optional): Add Jaeger as a third backend

This step demonstrates true 3-way fanout: console + LangSmith + Jaeger. Requires docker.

Skip this step if you don't have docker. Steps 8-11 work without it.

To run Jaeger locally:

```bash
docker run -d --name jaeger \
  -p 16686:16686 \
  -p 4318:4318 \
  jaegertracing/all-in-one:latest
```

Port 16686 is the UI; port 4318 is the OTLP HTTP endpoint. After it's running, the cell below adds a third `BatchSpanProcessor` pointing at the local Jaeger.

In [ ]:
# Uncomment the lines below if you have Jaeger running on localhost:4318

# jaeger_exporter = OTLPSpanExporter(endpoint="http://localhost:4318/v1/traces")
# provider.add_span_processor(BatchSpanProcessor(jaeger_exporter))
#
# # Run the agent once more — now spans land in console + LangSmith + Jaeger
# result = run_agent("Same task, three backends.")
#
# print("→ Open Jaeger UI at http://localhost:16686")
# print("→ Search for service: lab-18-otel-portable")
# print("→ Same trace data; Jaeger renders as a flame-graph")

print("(Step 7 skipped — uncomment the code block above if you have Jaeger running)")


## Step 8: Read the same trace in multiple places

This is a UI / observation step, not a code step.

Open the same agent run in each place:
- **Console** (already in your notebook output) — raw JSON; immediate; all the attributes visible.
- **LangSmith UI** — `smith.langchain.com` → project `lab-18-opentelemetry`. Renders as a conversation; messages-view + timeline-view available; queryable by attributes.
- **Jaeger UI** (if you did Step 7) — `http://localhost:16686` → service `lab-18-otel-portable`. Renders as a flame-graph; latency-focused; queryable by service/operation.

**What's the same**: the span data. Every backend sees the same `gen_ai.system`, `gen_ai.request.model`, token usage, span kind, parent-child relationships.

**What's different**: the UI rendering. LangSmith renders LLM calls as conversation turns; Jaeger renders them as flame-graph segments; Datadog (if you'd added it) would render them as service-map nodes. Same data; different lenses.

This is the production value of OTel-native instrumentation: you instrument once, every backend you might use ingests the same data, you pick UIs based on which questions they answer well.

## Step 9: Wrap the supervisor as a `gen_ai.invoke_agent` span

The auto-instrumentation in Step 5 covers LLM calls. It doesn't cover the orchestration layer — the agent boundary itself, where the supervisor decides routing and the workers execute. To capture that layer, manually wrap the agent with a `gen_ai.invoke_agent` span.

`SpanKind.INTERNAL` because the agent is local code, not a remote call. The agent span wraps the LLM call spans as children, giving a complete picture of the agent's execution.

In [ ]:
def run_agent_instrumented(task: str, max_steps: int = 4) -> dict:
    """Same agent loop as run_agent, but wrapped in a gen_ai.invoke_agent span
    so the agent boundary is visible in the trace tree."""
    with tracer.start_as_current_span(
        "gen_ai.invoke_agent",
        kind=SpanKind.INTERNAL,
    ) as agent_span:
        agent_span.set_attribute("gen_ai.agent.name", "supervisor")
        agent_span.set_attribute("gen_ai.agent.description",
                                  "Routes between researcher and writer workers")
        agent_span.set_attribute("gen_ai.operation.name", "invoke_agent")

        completed = {}
        for step in range(max_steps):
            # Each routing decision is a child gen_ai.chat span via auto-instrumentation
            action = supervisor_route(task, completed)
            agent_span.add_event(f"step_{step}_action", {"action": action})

            if action == "researcher":
                completed["researcher"] = researcher(task)
            elif action == "writer":
                completed["writer"] = writer(
                    completed.get("researcher", {}).get("findings", "")
                )
            elif action == "done":
                break

        agent_span.set_attribute("gen_ai.agent.steps", len(completed))
        return completed


# Run with the agent boundary visible
result = run_agent_instrumented(task)

print("\nAgent output:")
print("─" * 60)
print(result.get("writer", "[no writer output]")[:300])
print()
print("→ In LangSmith UI: the trace now has a gen_ai.invoke_agent root,")
print("  with the supervisor's LLM calls nested as gen_ai.chat children.")


## Step 10: Lab 17 vs Lab 18 side-by-side

The two paths produce equivalent traces for the same agent. The differences are in setup cost, ecosystem fit, and lock-in.

| | Lab 17 (LangSmith-native) | Lab 18 (OTel-native) |
|---|---|---|
| **Setup** | 2 env vars (`LANGSMITH_TRACING=true`, `LANGSMITH_API_KEY`) | ~15 lines: TracerProvider, Resource, BatchSpanProcessor, OTLPSpanExporter, headers |
| **LangChain auto-tracing** | Built-in — env vars enable it | Needs `opentelemetry-instrumentation-langchain` to be installed and called |
| **Custom function tracing** | `@traceable` decorator | Manual span via `tracer.start_as_current_span` (or auto-instrumentors) |
| **UI features** | Native — messages view, dataset workflow, evaluators all work | Native — same UI; LangSmith ingests OTel into the same workflows |
| **Backend portability** | Locked to LangSmith — `@traceable` doesn't emit OTel | Any OTel-compatible backend — fanout to LangSmith + Datadog + Jaeger + Phoenix + … |
| **Lock-in cost** | Switching means rewriting all decoration | Switching means changing the exporter endpoint |
| **OTel ecosystem fit** | Works (LangSmith March 2026 SDK is OTel-native internally) | Works natively; standard everywhere |
| **Best for** | LangChain/LangGraph-heavy teams; single-vendor commitment acceptable | Multi-backend deployments; corporate observability already on OTel; long-term portability |

Both produce traces with the same gen_ai attributes. Both work with `agentevals` from Lab 17 (it operates on the trajectory, not the trace format). The decision is about *what you'd do if you needed to add a second backend or switch platforms in 18 months*.

## Step 11: Synthesis

What this lab built:

- **An OTel-instrumented agent** with three exporters (console + LangSmith + optional Jaeger), one TracerProvider, multiple BatchSpanProcessors.
- **Manual `gen_ai.chat` spans** with `SpanKind.CLIENT` and the canonical attribute set.
- **`OpenAIInstrumentor` auto-instrumentation** for hands-off LLM-call tracing.
- **An agent-level `gen_ai.invoke_agent` span** that wraps the orchestration layer.
- **Verified the same trace data lands in every backend** — console renders as JSON, LangSmith as conversation, Jaeger as flame-graph.

What this lab didn't cover (deferred to later modules):

- **Tail-based sampling at the OTel Collector layer.** Module 6.
- **OTel baggage for cost attribution.** Module 6.
- **`langsmith-collector-proxy` for production fanout management.** Module 6.
- **Online evaluator registration.** Module 4.
- **Drift detection on metric distributions.** Module 5.
- **Multi-turn (threaded) evaluation.** Module 7.

The decision for your project:

- **Pick Lab 17's path (LangSmith-native)** when you're in a single-ecosystem commitment, prototyping fast, and lock-in cost is acceptable for velocity. The setup is two env vars; the ecosystem fit is tight.
- **Pick Lab 18's path (OTel-native)** when you have existing observability infrastructure (corporate Datadog, self-hosted Grafana), expect to add backends over time, or need cross-team standardization. The setup is one-time; the portability is permanent.
- **Pick the hybrid** when you want LangSmith's UI features AND portability: OTel-native instrumentation + LangSmith's platform-native extensions (agentevals registration, dataset workflow). Best of both, slightly more code.

Most production teams land in one of these three. Which one depends on team scale, ecosystem commitment, and existing infrastructure — not on which platform is "better."

✓ **Module 3 complete.** Module 4 (online evaluation + tail-based sampling) in a future batch.
